In [1]:
import pandas as pd
import numpy as np

import requests
import json
import ollama

import mysql.connector
from mysql.connector import Error


# Configuración de Ollama
OLLAMA_MODEL = "llama3.2:latest"  # Cambia por el modelo que tengas instalado

# Ruta del CSV base
CSV_INPUT = "promos.csv"

In [2]:
df = pd.read_csv(CSV_INPUT)
print(f"Total de promos: {len(df)}")
df.head()

Total de promos: 61


,Nr de la promoción,Año de comienzo,Año de fin,Tipo,Modalidad,Nombre,Nombre de la mujer referente
0,1,2016,2017,PW,Full-time,Ada,Ada Lovelace
1,2,2017,2017,PW,Full-time,Borg,Anita Borg
2,3,2017,2018,PW,Full-time,Clarke,Joan Clarke
3,4,2018,2018,PW,Full-time,Dorcas,Dorcas Muthoni
4,5,2018,2019,PW,Full-time,Easley,Annie Easley



    Llama a Ollama para obtener información estructurada sobre una mujer referente.
    
    Devuelve un dict con:
    - pais_principal
    - campo_principal
    - periodo_historico
    - anio_nacimiento
    - anio_muerte
    - logros_clave
    
    Si falla, devuelve None.


In [4]:
system_prompt = (
        "Eres un asistente experto en historia de mujeres en ciencia y tecnología. "
        "Tu tarea es devolver SIEMPRE un JSON válido (sin texto adicional, sin markdown, sin comillas triples).")
    

In [5]:
def enriquecer_referente_con_ollama(nombre_completo):  
    user_prompt = f"""
Te doy el nombre de una mujer referente en ciencia/tecnología: "{nombre_completo}".

Devuélveme ÚNICAMENTE un JSON con los siguientes campos (sin texto fuera del JSON):

- pais_principal (string): país o región donde desarrolló su actividad principal (ej: "Estados Unidos", "España", "Francia", etc.)
- campo_principal (string breve): campo principal (ej: "informática", "matemáticas", "física", "biología", "astronomía", "activismo tecnológico", etc.)
- periodo_historico (string breve): época aproximada (ej: "siglo XIX", "siglo XX", "era espacial", "medieval", "siglo XXI", etc.)
- anio_nacimiento (number o null): año de nacimiento si se conoce
- anio_muerte (number o null): año de fallecimiento (null si sigue viva o no se sabe)
- logros_clave (string): resumen breve (2-3 frases máximo) de sus principales logros, en español

No escribas explicaciones, ni texto introductorio, ni markdown, ni ```json```. 
Solo el JSON.

Ejemplo de estructura (NO lo uses como contenido, solo como formato):

{{
  "pais_principal": "Estados Unidos",
  "campo_principal": "informática",
  "periodo_historico": "siglo XX",
  "anio_nacimiento": 1906,
  "anio_muerte": 1992,
  "logros_clave": "Pionera en programación. Desarrolló el primer compilador."
}}

Ahora devuélveme el JSON para: {nombre_completo}
    """.strip()
    
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ]
        )
    except Exception as e:
        print(f"[ERROR llamando a Ollama para '{nombre_completo}']: {e}")
        return None
    
    content = response["message"]["content"].strip()
    
    # Intentar parsear directamente todo el contenido como JSON
    try:
        result = json.loads(content)
        return result
    except Exception as e:
        print(f"[ERROR parseando JSON para '{nombre_completo}']: {e}")
        print("Contenido devuelto por el modelo:\n", content)
        return None

## Ejemplos de uso de IA en un flujo de trabajo: 
- Obtener una informción en el flujo de trabajo. 
- Dado una galleria con info adicional: extraer fotos mas vistas, mas valoradas: META API
- 

In [6]:
resultado_debug = enriquecer_referente_con_ollama("Ada Lovelace")
print(resultado_debug)

{'pais_principal': 'Reino Unido', 'campo_principal': 'matemáticas e informática', 'periodo_historico': 'siglo XIX', 'anio_nacimiento': 1815, 'anio_muerte': None, 'logros_clave': 'Considerada la primera programadora. Desarrolló algoritmos para el analizador de música de Charles Babbage'}


In [ ]:
# Creamos columnas nuevas vacías
df["pais_principal"] = None
df["campo_principal"] = None
df["periodo_historico"] = None
df["anio_nacimiento"] = None
df["anio_muerte"] = None
df["logros_clave"] = None

for idx, row in df.iterrows():
    nombre_completo = row["Nombre de la mujer referente"]
    print(f"Procesando: {nombre_completo}...")
    
    info = enriquecer_referente_con_ollama(nombre_completo)
    
    if info is None:
        print(f"  ⚠️ No se pudo obtener información para '{nombre_completo}'.")
        continue
    # .at[] es un método de pandas para acceder/modificar una celda específica de forma rápida.
    df.at[idx, "pais_principal"] = info.get("pais_principal")
    df.at[idx, "campo_principal"] = info.get("campo_principal")
    df.at[idx, "periodo_historico"] = info.get("periodo_historico")
    df.at[idx, "anio_nacimiento"] = info.get("anio_nacimiento")
    df.at[idx, "anio_muerte"] = info.get("anio_muerte")
    df.at[idx, "logros_clave"] = info.get("logros_clave")
    
    print(f"  ✅ OK: {info.get('campo_principal')} | {info.get('pais_principal')}")

Procesando: Ada Lovelace...
  ✅ OK: matemáticas e informática | Inglaterra
Procesando: Anita Borg...
  ✅ OK: informática | Estados Unidos
Procesando: Joan Clarke...
  ✅ OK: cifra | Reino Unido
Procesando: Dorcas Muthoni...
  ✅ OK: activismo tecnológico | Kenia
Procesando: Annie Easley...
  ✅ OK: informática y física | Estados Unidos
Procesando: Mary Fairfax...
  ✅ OK: comercio y finanzas | Estados Unidos
Procesando: Grace Hopper...
  ✅ OK: informática | Estados Unidos
Procesando: Margaret Hamilton...
[ERROR parseando JSON para 'Margaret Hamilton']: Expecting ',' delimiter: line 7 column 154 (char 314)
Contenido devuelto por el modelo:
 {
  "pais_principal": "Estados Unidos",
  "campo_principal": "informática",
  "periodo_historico": "siglo XX",
  "anio_nacimiento": 1936,
  "anio_muerte": null,
  "logros_clave": "Desarrolló el sistema de control de la nave espacial Apollo. Fue pionera en la automatización del software y la gestión de proyectos."
  ⚠️ No se pudo obtener información para 

In [9]:
df.head()

,Nr de la promoción,Año de comienzo,Año de fin,Tipo,Modalidad,Nombre,Nombre de la mujer referente,pais_principal,campo_principal,periodo_historico,anio_nacimiento,anio_muerte,logros_clave
0,1,2016,2017,PW,Full-time,Ada,Ada Lovelace,None,None,None,None,None,None
1,2,2017,2017,PW,Full-time,Borg,Anita Borg,Estados Unidos,informática y activismo tecnológico,era de la informática moderna,None,None,"Desarrolló el primer compilador, luchó por la ..."
2,3,2017,2018,PW,Full-time,Clarke,Joan Clarke,None,None,None,None,None,None
3,4,2018,2018,PW,Full-time,Dorcas,Dorcas Muthoni,Kenia,biología,era moderna,1978,None,Investigadora en genética y bioingenieria. Des...
4,5,2018,2019,PW,Full-time,Easley,Annie Easley,Estados Unidos,ingeniería aeroespacial y matemáticas,siglo XX,1934,None,Contribuyó al desarrollo de cohetes y misiones...


In [10]:
df.to_csv("promos_referentes_enriquecido.csv", index=False, encoding="utf-8")

In [11]:
try:
    cnx = mysql.connector.connect(
        host='127.0.0.1',
        user='root',
        password='',
    )
    print('Conexión exitosa')
except Error as e:
    print('Error al conectar:', e)

Conexión exitosa


In [12]:
try:
    mycursor = cnx.cursor()
    mycursor.execute("CREATE DATABASE IF NOT EXISTS mujeres_referentes_ollama")
    print("Query exitosa")
except:
    print("Error.")

Query exitosa


In [13]:
# Usar la base de datos
mycursor.execute("USE mujeres_referentes_ollama")

# Crear la tabla si no existe
query = """
CREATE TABLE IF NOT EXISTS mujeres_referentes (
    id INT AUTO_INCREMENT PRIMARY KEY,
    nr_promocion INT,
    anio_comienzo INT,
    anio_fin INT,
    tipo VARCHAR(10),
    modalidad VARCHAR(20),
    nombre VARCHAR(50),
    nombre_completo VARCHAR(100),
    pais_principal VARCHAR(100),
    campo_principal VARCHAR(100),
    periodo_historico VARCHAR(50),
    anio_nacimiento INT,
    anio_muerte INT,
    logros_clave TEXT
)
"""

mycursor.execute(query)
cnx.commit()

print("✅ Base de datos 'mujeres_referentes' creada.")
print("✅ Tabla 'mujeres_referentes' lista.")

# Cerrar la conexión


✅ Base de datos 'mujeres_referentes' creada.
✅ Tabla 'mujeres_referentes' lista.


In [15]:
# Query de inserción
query = """
INSERT INTO mujeres_referentes (
    nr_promocion,
    anio_comienzo,
    anio_fin,
    tipo,
    modalidad,
    nombre,
    nombre_completo,
    pais_principal,
    campo_principal,
    periodo_historico,
    anio_nacimiento,
    anio_muerte,
    logros_clave
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

# Preparar lista de tuplas con todos los valores

df_clean = df.replace({np.nan: None, 'nan': None, 'NaN': None})

valores_lista = []
for idx, row in df.iterrows():
    valores = (
        int(row["Nr de la promoción"]),
        int(row["Año de comienzo"]),
        int(row["Año de fin"]),
        row["Tipo"],
        row["Modalidad"],
        row["Nombre"],
        row["Nombre de la mujer referente"],
        row["pais_principal"],
        row["campo_principal"],
        row["periodo_historico"],
        int(row["anio_nacimiento"]) if pd.notna(row["anio_nacimiento"]) else None,
        int(row["anio_muerte"]) if pd.notna(row["anio_muerte"]) else None,
        row["logros_clave"]
    )
    valores_lista.append(valores)

# Insertar todas las filas de una vez
mycursor.executemany(query, valores_lista)
cnx.commit()

print(f"🎉 Total de filas insertadas: {mycursor.rowcount}")



🎉 Total de filas insertadas: 61


In [16]:
# O contar cuántas hay en total
mycursor.execute("SELECT COUNT(*) FROM mujeres_referentes")
total = mycursor.fetchone()[0]
print(f"\nTotal de registros en la tabla: {total}")


Total de registros en la tabla: 122
